In [1]:
!pip install chess==1.10.0 einops==0.8.0 gdown==5.2.0 numpy==2.1.3 pandas==2.2.3 pyzstd==0.15.9 Requests==2.32.3 tqdm==4.65.0 maia2

  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully uninstalled tqdm-4.67.1


In [2]:
!pip install torch torchvision torchaudio


In [10]:
import torch
from maia2 import model, inference
import pandas as pd
import numpy as np
from tqdm import tqdm
import chess
import os

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
maia2_model = model.from_pretrained(type="rapid", device=device)
maia2_model.eval()

dummy_input = torch.randn(2, 18, 8, 8, device=device)
dummy_elo = torch.zeros(2, device=device).long()

with torch.no_grad():
    out = maia2_model(dummy_input, elos_self=dummy_elo, elos_oppo=dummy_elo)

print(f"Output is a tuple of length: {len(out)}")
for i, tensor in enumerate(out):
    print(f"Index {i} Tensor Shape: {tensor.shape}")


Model for rapid games already downloaded.
Model for rapid games loaded to cpu.
Output is a tuple of length: 3
Index 0 Tensor Shape: torch.Size([2, 1880])
Index 1 Tensor Shape: torch.Size([2, 2021])
Index 2 Tensor Shape: torch.Size([2])


In [ ]:
for name, child in maia2_model.named_children():
    print(name)

chess_cnn
to_patch_embedding
transformer
fc_1
fc_2
fc_3
fc_3_1
elo_embedding
dropout
last_ln


In [7]:
class FeatureExtractor:
    def __init__(self, model):
        self.model = model
        self.latent_vector = None
        self.hook = self.model.last_ln.register_forward_hook(self.save_output)

    def save_output(self, module, input, output):
        self.latent_vector = output

    def __call__(self, boards, elo_self, elo_oppo):
        self.model(boards, elos_self=elo_self, elos_oppo=elo_oppo)
        return self.latent_vector

def process_dataset(df, elo_indices=[0], batch_size=64):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    maia2_model = model.from_pretrained(type="rapid", device=device)
    maia2_model.to(device)
    maia2_model.eval()
    extractor = FeatureExtractor(maia2_model)
    fens = df['FEN'].tolist()
    num_puzzles = len(fens)
    num_elos = len(elo_indices)
    
    # (Puzzles, Elos, 1024 features)
    all_results = np.zeros((num_puzzles, num_elos, 1024), dtype=np.float32)
    
    for i in tqdm(range(0, num_puzzles, batch_size)):
        batch_end = min(i + batch_size, num_puzzles)
        batch_fens = fens[i:batch_end]
        
        batch_tensors = [inference.board_to_tensor(chess.Board(f)) for f in batch_fens]
        boards = torch.stack(batch_tensors).to(device)
        
        for elo_idx_in_list, skill_val in enumerate(elo_indices):
            elos = torch.full((len(batch_fens),), skill_val, device=device).long()
            
            with torch.no_grad():
                latent = extractor(boards, elos, elos)
                all_results[i:batch_end, elo_idx_in_list, :] = latent.cpu().numpy()
                
    return all_results

In [ ]:
elo_dict = inference.create_elo_dict()

ratings_to_test = [500,1100,1250, 1300, 1500, 1700, 1900, 2100, 2300, 2500, 3000]
for r in ratings_to_test:
    idx = inference.map_to_category(r, elo_dict)
    print(f"Rating {r} -> Index {idx}")

Rating 500 -> Index 0
Rating 1100 -> Index 1
Rating 1250 -> Index 2
Rating 1300 -> Index 3
Rating 1500 -> Index 5
Rating 1700 -> Index 7
Rating 1900 -> Index 9
Rating 2100 -> Index 10
Rating 2300 -> Index 10
Rating 2500 -> Index 10
Rating 3000 -> Index 10


In [5]:
df = pd.read_csv('../data/puzzle50000.csv')

In [8]:
feat = process_dataset(df, [i for i in range(0, 11)])

Model for rapid games already downloaded.
Model for rapid games loaded to cuda.


100%|██████████| 782/782 [00:28<00:00, 26.99it/s]


In [11]:
output_dir = "../features"

features_file = os.path.join(output_dir, "maia2_latent_features.npy")
np.save(features_file, feat)

ids_file = os.path.join(output_dir, "maia2_puzzle_ids.npy")
np.save(ids_file, df['PuzzleId'].values)

print(f"Success! Features saved to: {features_file}")
print(f"IDs saved to: {ids_file}")

Success! Features saved to: ../features/maia2_latent_features.npy
IDs saved to: ../features/maia2_puzzle_ids.npy


In [12]:
output_dir = "../latentFeatures"
os.makedirs(output_dir, exist_ok=True)

feat_subset = feat[:1000, :, :]
ids_subset = df['PuzzleId'].values[:1000]

np.save(os.path.join(output_dir, "maia2_latent_subset_2k.npy"), feat_subset)
np.save(os.path.join(output_dir, "maia2_subset_2k_ids.npy"), ids_subset)